## Loading and testing.

Reload runtime to save memory.

In [ ]:
!pip -q install -U "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
                 "trl>=0.11.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "safetensors>=0.4.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.8 MB/s eta 0:00:00


In [ ]:
# if running with colab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import gc
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch
import pandas as pd
import math
from tqdm import tqdm
import json
import random
from pathlib import Path

BETA = 0.1


# Paste directories here as needed

BASE_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# Where results will be stored
RESULTS_DIR = Path("/content/drive/MyDrive/DPO/eval_results_3594")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Country codes for adapters/data to evaluate. Add/remove codes here as needed.
# Use carefully. Takes ~ 1 hr per adapter run
# So for 2 countries it would take around 4 hrs if EVALUATE_ON_SELF, ~2 hrs if not
ADAPTER_COUNTRY_CODES = ["USA", "MEX"]
DATA_COUNTRY_CODES = ["USA", "MEX"]

# Whether to evaluate each adapter on the eval data for the same country.
# If False, only cross-country evaluations are run.
EVALUATE_ON_SELF = True

# Optional mapping for cases where path names use a different country code.
PATH_COUNTRY_CODES = {
    "USA": "US",
    "MEX": "MEX",
}

# path to eval/train files
DATA_DIR = Path("/content/drive/MyDrive/DPO")
# path to raw files
DATA_DIR1 = Path("/content/drive/MyDrive/DPO/pre-processing")

# Where adapters are stored
ADAPTER_DIRS = {
    country: f"/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_3594_{PATH_COUNTRY_CODES.get(country, country)}_3"
    for country in ADAPTER_COUNTRY_CODES
}

EVAL_FILES = {
    country: DATA_DIR / f"{country}_3594_eval_3.jsonl"
    for country in DATA_COUNTRY_CODES
}

RAW_FILES = {
    country: DATA_DIR1 / f"{PATH_COUNTRY_CODES.get(country, country)}_triplets_3594_3.jsonl"
    for country in DATA_COUNTRY_CODES
}

TRAIN_FILES = {
    country: DATA_DIR / f"{country}_3594_train_3.jsonl"
    for country in DATA_COUNTRY_CODES
}

BETA = 0.1


In [ ]:
# login with hugging face.
# must have an account with them
# occasionally also requests an access token
#!pip install -U "huggingface_hub[cli]"
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def format_prompt_text(tokenizer, prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(prompt_text)}],
        tokenize=False,
        add_generation_prompt=True,
    )

In [ ]:
def get_model_device(model):
    """
    Returns the device where the model is located.
    Works for regular models and PEFT/QLoRA models.
    """
    return next(model.parameters()).device


@torch.no_grad()
def sequence_logprob(
    model,
    tokenizer,
    prompt_text,
    completion_text,
    max_prompt_tokens=512,
    max_completion_tokens=256,
):
    """
    Computes log p(completion | prompt) for a single prompt-completion pair.

    Important:
    This should use the same prompt formatting as DPO training.
    The log probability is summed over completion tokens only.
    """

    model.eval()
    device = get_model_device(model)

    # Use the same chat-template format used during DPO training.

    formatted_prompt = format_prompt_text(tokenizer, prompt_text)

    prompt_ids = tokenizer(
        formatted_prompt,
        add_special_tokens=False,
    ).input_ids

    completion_ids = tokenizer(
        completion_text,
        add_special_tokens=False,
    ).input_ids

    # Truncate if needed.
    if len(prompt_ids) > max_prompt_tokens:
        prompt_ids = prompt_ids[-max_prompt_tokens:]

    if len(completion_ids) > max_completion_tokens:
        completion_ids = completion_ids[:max_completion_tokens]

    input_ids = prompt_ids + completion_ids

    # Mask prompt tokens so only completion tokens count in the loss/logprob.
    labels = [-100] * len(prompt_ids) + completion_ids

    input_ids = torch.tensor([input_ids], device=device)
    labels = torch.tensor([labels], device=device)

    outputs = model(input_ids=input_ids)
    logits = outputs.logits

    # Shift for next-token prediction.
    shifted_logits = logits[:, :-1, :]
    shifted_labels = labels[:, 1:]

    log_probs = torch.log_softmax(shifted_logits, dim=-1)

    mask = shifted_labels.ne(-100)

    safe_labels = shifted_labels.clone()
    safe_labels[~mask] = 0

    token_log_probs = log_probs.gather(
        dim=-1,
        index=safe_labels.unsqueeze(-1),
    ).squeeze(-1)

    completion_logprob = (token_log_probs * mask).sum()

    return float(completion_logprob.detach().cpu())

def dpo_implied_reward_delta(
    adapter_model,
    tokenizer,
    prompt,
    chosen,
    rejected,
    beta=BETA,
):
    """
    Computes the DPO implied reward difference using the same PEFT model:

        ref logprobs: adapter disabled
        adapter logprobs: adapter enabled

    This avoids accidentally comparing the adapter to itself.
    """

    # Reference model: same base, adapter disabled.
    with adapter_model.disable_adapter():
        ref_chosen_logp = sequence_logprob(
            adapter_model,
            tokenizer,
            prompt,
            chosen,
        )

        ref_rejected_logp = sequence_logprob(
            adapter_model,
            tokenizer,
            prompt,
            rejected,
        )

    # Adapter model: adapter enabled.
    adapter_chosen_logp = sequence_logprob(
        adapter_model,
        tokenizer,
        prompt,
        chosen,
    )

    adapter_rejected_logp = sequence_logprob(
        adapter_model,
        tokenizer,
        prompt,
        rejected,
    )

    ref_margin = ref_chosen_logp - ref_rejected_logp
    adapter_margin = adapter_chosen_logp - adapter_rejected_logp

    reward_delta = beta * (adapter_margin - ref_margin)

    dpo_pref_prob = 1.0 / (1.0 + math.exp(-reward_delta))

    return {
        "ref_chosen_logp": ref_chosen_logp,
        "ref_rejected_logp": ref_rejected_logp,
        "adapter_chosen_logp": adapter_chosen_logp,
        "adapter_rejected_logp": adapter_rejected_logp,
        "ref_margin": ref_margin,
        "adapter_margin": adapter_margin,
        "dpo_reward_delta": reward_delta,
        "dpo_pref_prob": dpo_pref_prob,
        "dpo_prefers_chosen": reward_delta > 0,
    }

@torch.no_grad()
def generate_model_answer(
    model,
    tokenizer,
    prompt_text,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
):
    """
    Generates a free-form answer from the model for a prompt.
    Uses the same prompt format as reward recovery and training.
    """

    model.eval()
    device = get_model_device(model)

    formatted_prompt = format_prompt_text(tokenizer, prompt_text)

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Keep only newly generated tokens, not the prompt.
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return generated_text

def evaluate_adapter_reward_recovery(
    adapter_model,
    tokenizer,
    eval_file,
    model_name,
    eval_country,
    beta=BETA,
    # CHANGE HERE FOR TEST OUTPUT
    max_examples=200,
    #max_examples=5,
    generate_answers=True,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
):
    """
    Runs DPO implied reward recovery on a held-out eval JSONL file.

    Also optionally generates a free-form answer from the adapter
    for each prompt and stores it in the result dataframe.
    """

    rows = load_jsonl(eval_file)

    if max_examples is not None:
        rows = rows[:max_examples]

    results = []

    for ex in tqdm(rows, desc=f"Reward recovery: {model_name} on {eval_country}"):
        out = dpo_implied_reward_delta(
            adapter_model=adapter_model,
            tokenizer=tokenizer,
            prompt=ex["prompt"],
            chosen=ex["chosen"],
            rejected=ex["rejected"],
            beta=beta,
        )

        if generate_answers:
            generated_answer = generate_model_answer(
                model=adapter_model,
                tokenizer=tokenizer,
                prompt_text=ex["prompt"],
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )
        else:
            generated_answer = None

        results.append({
            "model": model_name,
            "eval_country": eval_country,
            "country": ex.get("country"),
            "item_id": ex.get("item_id"),
            #"dimension": ex.get("dimension"),
            "gps_dimension": ex.get("gps_dimension"),
            "prompt": ex["prompt"],
            "chosen": ex["chosen"],
            "rejected": ex["rejected"],
            "generated_answer": generated_answer,
            **out,
        })

    return pd.DataFrame(results)



In [ ]:

def load_base_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )

    base_model.eval()

    return base_model, tokenizer


def load_adapter_model(adapter_dir):
    base_model, tokenizer = load_base_model_and_tokenizer()

    adapter_model = PeftModel.from_pretrained(
        base_model,
        adapter_dir,
    )

    adapter_model.eval()

    return adapter_model, tokenizer


def cleanup_models(*models):
    for model in models:
        del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def split_country_file(raw_path, train_path, eval_path, country, train_frac=0.80, seed=42):
    rows = load_jsonl(raw_path)

    # Add useful metadata for later validation.
    for i, row in enumerate(rows):
        row.setdefault("country", country)
        row.setdefault("item_id", f"{country}_{i:04d}")

    rng = random.Random(seed)
    rng.shuffle(rows)

    n_train = int(len(rows) * train_frac)

    train_rows = rows[:n_train]
    eval_rows = rows[n_train:]

    write_jsonl(train_rows, train_path)
    write_jsonl(eval_rows, eval_path)

    print(f"{country}: {len(rows)} total")
    print(f"  train: {len(train_rows)} -> {train_path}")
    print(f"  eval:  {len(eval_rows)} -> {eval_path}")

    return train_rows, eval_rows


In [ ]:
# Load one adapter for the smoke test.
# The full cross-evaluation loop below loads and cleans up each adapter one at a time.
#SMOKE_TEST_ADAPTER_COUNTRY = ADAPTER_COUNTRY_CODES[0]
#smoke_model, tokenizer = load_adapter_model(ADAPTER_DIRS[SMOKE_TEST_ADAPTER_COUNTRY])


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
def smoke_generate_once(model, tokenizer, prompt, max_new_tokens=80):
    was_training = model.training
    model.eval()

    messages = [{"role": "user", "content": prompt}]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    ]
    terminators = [
        t for t in terminators
        if t is not None and t != tokenizer.unk_token_id
    ]

    with torch.no_grad():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=terminators,
            use_cache=True,
        )

    prompt_len = encoded["input_ids"].shape[-1]
    generated_ids = output_ids[0][prompt_len:]

    text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    print("\n[smoke test]")
    print(text)

    if "gibberish_score" in globals():
        print("gibberish_score:", gibberish_score(text))

    if was_training:
        model.train()

    return text

In [ ]:
"""
smoke_generate_once(
    smoke_model,
    tokenizer,
    "Explain in one paragraph why housing affordability matters.\n\nAnswer:"
)
"""

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



[smoke test]
Housing affordability matters because it has a significant impact on individuals' and families' financial stability, well-being, and quality of life. When housing costs are too high relative to income, people may struggle to afford basic necessities like food, healthcare, and education, leading to increased stress, debt, and even homelessness. Unaffordable housing can also limit social mobility, as those who cannot afford decent housing


"Housing affordability matters because it has a significant impact on individuals' and families' financial stability, well-being, and quality of life. When housing costs are too high relative to income, people may struggle to afford basic necessities like food, healthcare, and education, leading to increased stress, debt, and even homelessness. Unaffordable housing can also limit social mobility, as those who cannot afford decent housing"

In [ ]:
# ============================
# Cross-evaluate adapters
# ============================

all_adapter_dfs = []

for adapter_country in ADAPTER_COUNTRY_CODES:
    adapter_dir = ADAPTER_DIRS[adapter_country]
    adapter_model, tokenizer = load_adapter_model(adapter_dir)

    adapter_eval_dfs = []

    for eval_country, eval_file in EVAL_FILES.items():
        if not EVALUATE_ON_SELF and adapter_country == eval_country:
            continue

        df_adapter_on_eval = evaluate_adapter_reward_recovery(
            adapter_model=adapter_model,
            tokenizer=tokenizer,
            eval_file=eval_file,
            model_name=f"{adapter_country}_adapter",
            eval_country=eval_country,
            beta=BETA,
            generate_answers=True,
            max_new_tokens=120,
            temperature=0.7,
            top_p=0.9,
        )

        df_adapter_on_eval.to_csv(
            RESULTS_DIR / f"reward_recovery_{adapter_country}_adapter_3594_on_{eval_country}_3.csv",
            index=False,
        )

        adapter_eval_dfs.append(df_adapter_on_eval)
        all_adapter_dfs.append(df_adapter_on_eval)

    if adapter_eval_dfs:
        df_adapter = pd.concat(
            adapter_eval_dfs,
            ignore_index=True,
        )

        df_adapter.to_csv(
            RESULTS_DIR / f"reward_recovery_{adapter_country}_adapter_3594_3.csv",
            index=False,
        )

    cleanup_models(adapter_model)
    del adapter_model, tokenizer

if all_adapter_dfs:
    df_all_adapters = pd.concat(
        all_adapter_dfs,
        ignore_index=True,
    )

    df_all_adapters.to_csv(
        RESULTS_DIR / "reward_recovery_all_adapters_3594_3.csv",
        index=False,
    )

    df_all_adapters.head()



Reward recovery: US_adapter on US:   0%|          | 0/200 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.

Reward recovery: US_adapter on US: 100%|██████████| 200/200 [49:48<00:00, 14.94s/it]

Reward recovery: US_adapter on Mexico: 100%|██████████| 200/200 [47:40<00:00, 14.30s/it]


,model,eval_country,country,item_id,gps_dimension,prompt,chosen,rejected,generated_answer,ref_chosen_logp,ref_rejected_logp,adapter_chosen_logp,adapter_rejected_logp,ref_margin,adapter_margin,dpo_reward_delta,dpo_pref_prob,dpo_prefers_chosen
0,US_adapter,US,US,US_2212,altruism,Context: I'm a member of a community garden an...,"I think about sharing my seeds and tools, whic...",I decide to offer the seeds and tools to the n...,I think it would be great to offer the new gar...,-101.0,-86.5,-96.0,-71.0,-14.5,-25.0,-1.05,0.259225,False
1,US_adapter,US,US,US_1309,negative_reciprocity,"Context: I'm part of a carpool, and one member...",I'll bring up the issue but keep it light. Whi...,I'll bring up the issue but in a gentle way. I...,While it's frustrating to feel like I'm consis...,-125.0,-116.0,-112.0,-102.0,-9.0,-10.0,-0.10,0.475021,False
2,US_adapter,US,US,US_1375,altruism,Context: After a natural disaster hits my regi...,I decide to keep the room for my work. While i...,I decide to keep the room for my home office. ...,While I feel a strong sense of responsibility ...,-98.0,-79.0,-125.0,-99.0,-19.0,-26.0,-0.70,0.331812,False
3,US_adapter,US,US,US_1112,positive_reciprocity,"Context: In an online forum, a stranger spends...",I take a moment to consider the time they spen...,I allocate an hour to thoroughly review their ...,"It's a nice gesture to review their project, b...",-94.0,-67.0,-93.0,-79.5,-27.0,-13.5,1.35,0.794130,True
4,US_adapter,US,US,US_0960,altruism,Context: I've just finished a project and have...,"I choose to donate the supplies to the school,...",I choose to keep the supplies because I might ...,While it's tempting to hold onto the supplies ...,-103.5,-101.0,-117.0,-135.0,-2.5,18.0,2.05,0.885948,True
